In [15]:
# Import required libraries
import pandas as pd
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import string

# Load data
df = pd.read_csv('/Users/janhavijadhav/Downloads/data_stories_one_shot.csv')

# Step 1: Clean text (lowercase + remove punctuation)
def basic_clean(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation))

df['processed'] = df['Sentence'].apply(basic_clean)

# Step 2: Label encoding
df['label'] = df['Stage'].apply(lambda x: 'Show' if x == 1 else 'Tell')
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

# Step 3: Define classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear'),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier()
}

# Step 4: TF-IDF with stop word removal and bigrams
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

# Step 5: 5-Fold Cross-Validation
results_cv = {}
for name, clf in classifiers.items():
    pipeline = Pipeline([
        ('tfidf', vectorizer),
        ('model', clf)
    ])
    scores = cross_val_score(pipeline, df['processed'], df['label_encoded'], cv=5, scoring='accuracy')
    results_cv[name] = scores.mean()

# Step 6: Leave-One-Plot-Out CV (Logistic Regression only)
group_kfold = GroupKFold(n_splits=len(df['Plot_Name'].unique()))
pipeline_lr = Pipeline([
    ('tfidf', vectorizer),
    ('model', LogisticRegression(max_iter=1000))
])
lopo_scores = cross_val_score(
    pipeline_lr,
    df['processed'],
    df['label_encoded'],
    cv=group_kfold.split(df['processed'], df['label_encoded'], groups=df['Plot_Name'])
)

# Step 7: Output results
print("5-Fold Cross-Validation Accuracy:")
for name, acc in results_cv.items():
    print(f"{name}: {acc:.4f}")

print(f"\nLeave-One-Plot-Out Accuracy (Logistic Regression): {lopo_scores.mean():.4f}")


5-Fold Cross-Validation Accuracy:
Logistic Regression: 0.6000
SVM: 0.8077
Naive Bayes: 0.7231
Random Forest: 0.6077

Leave-One-Plot-Out Accuracy (Logistic Regression): 0.6335


In [19]:
# Import libraries
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import string

# Load dataset
df = pd.read_csv('/Users/janhavijadhav/Downloads/data_stories_one_shot.csv')

# Clean text: lowercase + punctuation removal
df['processed'] = df['Sentence'].apply(lambda x: x.lower().translate(str.maketrans('', '', string.punctuation)))

# Encode labels: Show vs Tell
df['label'] = df['Stage'].apply(lambda x: 'Show' if x == 1 else 'Tell')
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear'),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier()
}

# TF-IDF Vectorizer: use unigrams, remove stop words, sublinear tf
vectorizer = TfidfVectorizer(stop_words='english', sublinear_tf=True)

# 5-Fold Stratified Cross-Validation
results_cv = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in models.items():
    pipeline = Pipeline([
        ('tfidf', vectorizer),
        ('model', clf)
    ])
    scores = cross_val_score(pipeline, df['processed'], df['label_encoded'], cv=skf, scoring='accuracy')
    results_cv[name] = scores.mean()

# Leave-One-Plot-Out CV using Logistic Regression
group_kfold = GroupKFold(n_splits=len(df['Plot_Name'].unique()))
pipeline_lr = Pipeline([
    ('tfidf', vectorizer),
    ('model', LogisticRegression(max_iter=1000))
])
scores_lopo = cross_val_score(
    pipeline_lr,
    df['processed'],
    df['label_encoded'],
    cv=group_kfold.split(df['processed'], df['label_encoded'], groups=df['Plot_Name'])
)

# Output results
print("5-Fold Cross-Validation Accuracy (Stratified):")
for model_name, acc in results_cv.items():
    print(f"{model_name}: {acc:.4f}")

print(f"\n Leave-One-Plot-Out Accuracy (Logistic Regression): {scores_lopo.mean():.4f}")


5-Fold Cross-Validation Accuracy (Stratified):
Logistic Regression: 0.7308
SVM: 0.7846
Naive Bayes: 0.7769
Random Forest: 0.7308

 Leave-One-Plot-Out Accuracy (Logistic Regression): 0.6672
